# De-identified NSCLC patient records with PET/CT SUVmax, PD-L1 immunohistochemistry, lesion site, and clinicopathological variables from a single-centre cohort, 2021-2024 Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, inspecting, and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

- `https://sen.science/doi/10.71728/senscience.b0r5-4r6b/fair2.json`

### Goals
In this notebook, we will:
- load the Croissant metadata with `mlcroissant`
- inspect available record sets and their `@id` values
- load records into pandas DataFrames
- perform basic exploratory data analysis
- create a few useful visualizations

> Important: all dataset entities should be referenced by their Croissant `@id` values whenever available.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant` and inspect the high-level dataset description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint

sns.set_theme(style="whitegrid")

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.b0r5-4r6b/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Access metadata as a single object
print("Dataset name:", getattr(metadata, "name", "N/A"))
print("Description:", getattr(metadata, "description", "N/A"))
print("License:", getattr(metadata, "license", "N/A"))
print("Version:", getattr(metadata, "version", "N/A"))
print("URL:", croissant_url)

In [ ]:
# Convert metadata to JSON for easier structured inspection when needed
metadata_json = metadata.to_json()

print("Top-level metadata keys:")
print(sorted(metadata_json.keys()))

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Because Croissant datasets can expose one or more record sets, we first inspect the metadata structure and then list all available record set identifiers.

In [ ]:
# Inspect record sets declared in the Croissant metadata JSON
record_sets_metadata = metadata_json.get("recordSet", [])

print(f"Number of record sets declared in metadata: {len(record_sets_metadata)}")

for i, rs in enumerate(record_sets_metadata):
    rs_id = rs.get("@id", f"record_set_{i}")
    rs_name = rs.get("name", "N/A")
    print(f"\nRecord set {i+1}")
    print("  @id :", rs_id)
    print("  name:", rs_name)
    print("  keys:", sorted(rs.keys()))
    
    fields = rs.get("field", [])
    print(f"  field count: {len(fields)}")
    for field in fields:
        field_id = field.get("@id", "N/A")
        field_name = field.get("name", "N/A")
        field_type = field.get("dataType", "N/A")
        print(f"    - field @id: {field_id} | name: {field_name} | dataType: {field_type}")

### Programmatic preview of records
If record sets are available, we can preview records directly using `dataset.records(record_set=<record_set_@id>)`.

The next cell is written defensively so the notebook remains usable even if the dataset exposes zero record sets through the current metadata endpoint.

In [ ]:
record_set_ids = [rs.get("@id") for rs in record_sets_metadata if rs.get("@id")]
print("Record set @id values:", record_set_ids)

if record_set_ids:
    preview_record_set_id = record_set_ids[0]
    print(f"\nPreviewing up to 3 records from record set: {preview_record_set_id}")
    
    for idx, record in enumerate(dataset.records(record_set=preview_record_set_id)):
        print(record)
        if idx >= 2:
            break
else:
    print("No record sets were declared in metadata, so there are no records to preview with dataset.records(...).")

## 3. Data Extraction
Load data from each available record set into a pandas DataFrame.

We use variables for record set identifiers so the notebook remains reusable across Croissant datasets.

In [ ]:
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set {record_set_id} with shape {df.shape}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

if not dataframes:
    print("No DataFrames were created because no record sets were available or loadable.")

In [ ]:
# Inspect columns for each loaded record set
for record_set_id, df in dataframes.items():
    print(f"\nRecord set @id: {record_set_id}")
    print("Columns:", df.columns.tolist())
    display(df.head())

if not dataframes:
    print("No tables to display.")

### Build a field reference table from metadata
This helper table makes it easier to inspect available field `@id` values and connect them to loaded DataFrame columns.

In [ ]:
field_rows = []

for rs in record_sets_metadata:
    rs_id = rs.get("@id", "N/A")
    for field in rs.get("field", []):
        field_rows.append({
            "record_set_@id": rs_id,
            "field_@id": field.get("@id", "N/A"),
            "field_name": field.get("name", "N/A"),
            "dataType": field.get("dataType", "N/A"),
            "description": field.get("description", "")
        })

field_reference_df = pd.DataFrame(field_rows)
print(f"Field reference table shape: {field_reference_df.shape}")
display(field_reference_df.head(20))

## 4. Exploratory Data Analysis (EDA)
In this section, we apply common preprocessing steps:
- identify numeric and categorical columns
- filter records on a numeric variable
- normalize a numeric variable
- group by a categorical field
- inspect missing values

The code below is dynamic and chooses columns based on the actual loaded data.

In [ ]:
if dataframes:
    # Select the first available record set for a generic EDA workflow
    active_record_set_id = next(iter(dataframes.keys()))
    active_df = dataframes[active_record_set_id].copy()
    
    print("Active record set @id:", active_record_set_id)
    print("Shape:", active_df.shape)
    
    # Attempt numeric conversion where possible
    converted_df = active_df.copy()
    for col in converted_df.columns:
        converted_df[col] = pd.to_numeric(converted_df[col], errors="ignore")
    
    numeric_columns = converted_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_columns = converted_df.select_dtypes(exclude=[np.number]).columns.tolist()
    
    print("Numeric columns:", numeric_columns)
    print("Categorical columns:", categorical_columns)
else:
    active_record_set_id = None
    active_df = pd.DataFrame()
    converted_df = pd.DataFrame()
    numeric_columns = []
    categorical_columns = []
    print("No data available for EDA.")

In [ ]:
# Missing value summary
if not converted_df.empty:
    missing_summary = converted_df.isna().sum().sort_values(ascending=False).to_frame("missing_count")
    missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(converted_df)) * 100
    display(missing_summary.head(20))
else:
    print("No data available to compute missing values.")

In [ ]:
# Filter, normalize, and group using dynamically selected columns
if not converted_df.empty and numeric_columns:
    numeric_field_id = numeric_columns[0]
    numeric_series = converted_df[numeric_field_id].dropna()
    
    if len(numeric_series) > 0:
        threshold = numeric_series.median()
        filtered_df = converted_df[converted_df[numeric_field_id] > threshold].copy()
        
        print(f"Using numeric field: {numeric_field_id}")
        print(f"Threshold (median): {threshold}")
        print(f"Filtered shape: {filtered_df.shape}")
        display(filtered_df.head())
        
        std_value = filtered_df[numeric_field_id].std()
        if pd.notna(std_value) and std_value != 0:
            filtered_df[f"{numeric_field_id}_normalized"] = (
                (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / std_value
            )
        else:
            filtered_df[f"{numeric_field_id}_normalized"] = 0.0
        
        print(f"Normalized field created: {numeric_field_id}_normalized")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        if categorical_columns:
            group_field_id = categorical_columns[0]
            grouped_df = (
                filtered_df.groupby(group_field_id)[numeric_field_id]
                .agg(["count", "mean", "median", "min", "max"])
                .sort_values("count", ascending=False)
            )
            print(f"Grouped summary by: {group_field_id}")
            display(grouped_df.head(10))
        else:
            print("No categorical column available for grouping.")
    else:
        print("Selected numeric column does not contain usable values.")
else:
    print("No numeric data available for filtering and normalization.")

### Descriptive statistics
For numeric variables, summary statistics help us understand range, central tendency, and spread.

In [ ]:
if not converted_df.empty and numeric_columns:
    display(converted_df[numeric_columns].describe().T)
else:
    print("No numeric columns available for descriptive statistics.")

## 5. Visualization
Visualize data distributions and simple relationships between fields.

The cells below automatically adapt to the available columns in the loaded record set.

In [ ]:
# Histogram of the first numeric field
if not converted_df.empty and numeric_columns:
    plot_numeric_field_id = numeric_columns[0]
    
    plt.figure(figsize=(8, 5))
    sns.histplot(converted_df[plot_numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {plot_numeric_field_id}")
    plt.xlabel(plot_numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric column available for histogram.")

In [ ]:
# Boxplot of a numeric field by a categorical field
if not converted_df.empty and numeric_columns and categorical_columns:
    plot_numeric_field_id = numeric_columns[0]
    plot_group_field_id = categorical_columns[0]
    
    temp_df = converted_df[[plot_numeric_field_id, plot_group_field_id]].dropna().copy()
    top_categories = temp_df[plot_group_field_id].astype(str).value_counts().head(10).index
    temp_df = temp_df[temp_df[plot_group_field_id].astype(str).isin(top_categories)]
    
    if not temp_df.empty:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=temp_df, x=plot_group_field_id, y=plot_numeric_field_id)
        plt.title(f"{plot_numeric_field_id} by {plot_group_field_id}")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("Not enough non-null grouped data to draw a boxplot.")
else:
    print("Need at least one numeric and one categorical column for a grouped boxplot.")

In [ ]:
# Scatter plot of the first two numeric fields, if available
if not converted_df.empty and len(numeric_columns) >= 2:
    x_field_id = numeric_columns[0]
    y_field_id = numeric_columns[1]
    
    plt.figure(figsize=(7, 5))
    sns.scatterplot(data=converted_df, x=x_field_id, y=y_field_id)
    plt.title(f"{y_field_id} vs {x_field_id}")
    plt.tight_layout()
    plt.show()
else:
    print("Need at least two numeric columns for a scatter plot.")

## 6. Conclusion
This notebook demonstrated a reusable Croissant-based workflow for dataset exploration with `mlcroissant`.

### Summary
- We loaded the dataset metadata from the Croissant URL.
- We inspected available record sets and field `@id` values.
- We attempted to extract all available record sets into pandas DataFrames.
- We applied dynamic EDA steps including missing value analysis, filtering, normalization, grouping, and descriptive statistics.
- We created visualizations that adapt to the structure of the loaded data.

### Notes
- This particular dataset metadata may expose zero record sets through the current Croissant endpoint, in which case record-level extraction will not produce tables.
- If record sets are added or become available later, the notebook will automatically use their `@id` values for loading and analysis.
- For production analysis, prefer explicit field `@id` references from the metadata rather than relying only on automatically selected columns.